# 03. 카탈로그 품질 점검

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Reasonofmoon/hangeorum-playground/blob/main/notebooks/03_catalog_quality_check.ipynb)

**점검 항목**
1. `media` 탭: 대체 텍스트(`alt`)가 빈 미디어
2. `steps` 탭: `expected` 또는 `recovery`가 빈 단계
3. `tutorials` 탭: 단계가 0개인 튜토리얼
4. `tutorials`, `steps`: 겹치는 `code`

읽기만 합니다. 시트를 고치지 않습니다. 결과 표를 보고 관리자 웹 앱에서 직접 고치세요.

In [ ]:
# 1) 내 Google 계정으로 인증합니다. 창이 뜨면 시트를 볼 수 있는 계정을 고르고 허용하세요.
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
print("인증 완료")

In [ ]:
#@title 2) 시트 ID 입력
#@markdown 시트 주소 `https://docs.google.com/spreadsheets/d/<여기>/edit` 의 `<여기>` 부분만 넣습니다.
SHEET_ID = "" #@param {type:"string"}
assert SHEET_ID.strip(), "SHEET_ID를 입력하세요."
sh = gc.open_by_key(SHEET_ID.strip())
print("열린 시트:", sh.title)

## 3) 탭 읽기

In [ ]:
import pandas as pd

def read_tab(name):
    try:
        return pd.DataFrame(sh.worksheet(name).get_all_records())
    except gspread.exceptions.WorksheetNotFound:
        print(f"'{name}' 탭이 없습니다.")
        return pd.DataFrame()

tutorials = read_tab("tutorials")
steps = read_tab("steps")
media = read_tab("media")
for n, d in [("tutorials", tutorials), ("steps", steps), ("media", media)]:
    print(f"{n}: {len(d)}행, 열 = {list(d.columns)}")

## 4) 점검 실행
열 이름 가정: `tutorials`의 ID 열은 `id`, `steps`의 튜토리얼 열은 `tutorial_id`, 두 탭 모두 `code` 열이 있음. 실제 열 이름이 다르면 아래 `TUT_ID_COL`, `STEP_TUT_COL`만 고치세요.
`media` 열: `media_id, drive_file_id, type, alt, caption, source, rights, uploaded_by, uploaded_at`

In [ ]:
TUT_ID_COL = "id"
STEP_TUT_COL = "tutorial_id"

def blank(series):
    return series.astype(str).str.strip().isin(["", "nan", "None"])

issues = []

# 1) 대체 텍스트 없는 미디어
if not media.empty:
    if "alt" in media:
        for _, r in media[blank(media["alt"])].iterrows():
            issues.append(("media", "대체 텍스트 없음", r.get("media_id", ""), f"type={r.get('type', '')}"))
    else:
        issues.append(("media", "'alt' 열 없음", "", ""))

# 2) expected / recovery 빈 단계
if not steps.empty:
    for col in ["expected", "recovery"]:
        if col in steps:
            for _, r in steps[blank(steps[col])].iterrows():
                issues.append(("steps", f"{col} 비어 있음", r.get("code", ""), f"{STEP_TUT_COL}={r.get(STEP_TUT_COL, '')}"))
        else:
            issues.append(("steps", f"'{col}' 열 없음", "", ""))

# 3) 단계 0개 튜토리얼
if not tutorials.empty and TUT_ID_COL in tutorials:
    if STEP_TUT_COL in steps:
        counts = steps[STEP_TUT_COL].astype(str).str.strip().value_counts()
    else:
        counts = pd.Series(dtype=int)
    for _, r in tutorials.iterrows():
        tid = str(r[TUT_ID_COL]).strip()
        if int(counts.get(tid, 0)) == 0:
            issues.append(("tutorials", "단계 0개", r.get("code", ""), f"{TUT_ID_COL}={tid}"))

# 4) 겹치는 code
for name, d in [("tutorials", tutorials), ("steps", steps)]:
    if not d.empty and "code" in d:
        c = d["code"].astype(str).str.strip()
        dup = c[(c != "") & c.duplicated(keep=False)]
        for v, n in dup.value_counts().items():
            issues.append((name, "code 중복", v, f"{n}번 나옴"))

report = pd.DataFrame(issues, columns=["탭", "문제", "code/ID", "참고"])
print("총 문제 수:", len(report))
report

## 5) 문제 종류별 개수

In [ ]:
if report.empty:
    print("문제 없음")
else:
    display(report.groupby(["탭", "문제"]).size().rename("개수").to_frame())

In [ ]:
# 6) (선택) 보고서를 CSV로 내려받기
from google.colab import files
report.to_csv("catalog_quality_report.csv", index=False, encoding="utf-8-sig")
files.download("catalog_quality_report.csv")

---
**고치는 순서 제안**
1. 대체 텍스트: 그림이 무엇을 보여 주는지 한 문장으로 씁니다.
2. expected / recovery: 성공하면 보이는 것, 실패하면 할 일을 씁니다.
3. 단계 0개: 단계를 추가하거나 튜토리얼을 비공개로 둡니다.
4. code 중복: 하나만 남기고 번호를 다시 매깁니다.